In [2]:
import pandas as pd
import numpy as np
import torch
import mlflow
import mlflow.pytorch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# MLflow 실험실 설정
mlflow.set_experiment("Tamagotchi_Emotion_Project")
print("✅ 라이브러리 로드 및 MLflow 설정 완료")

2026/02/24 08:17:34 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/02/24 08:17:34 INFO mlflow.store.db.utils: Updating database tables
2026/02/24 08:17:36 INFO mlflow.tracking.fluent: Experiment with name 'Tamagotchi_Emotion_Project' does not exist. Creating a new experiment.


✅ 라이브러리 로드 및 MLflow 설정 완료


In [4]:
# 1. 데이터 로드
df = pd.read_csv('tamagotchi_dataset.csv', encoding='utf-8-sig')

# 2. 감정 레이블링 (매핑 확인)
emotion_mapping = {'기쁨': 0, '슬픔': 1, '화남': 2, '평온': 3}
df['label'] = df['tamagotchi_emotion'].map(emotion_mapping)

# 3. 데이터 분할 (8:2)
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, shuffle=True)
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

# 4. 토크나이저 준비
model_name = "beomi/KcELECTRA-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=64)

# 5. 토크나이징 적용
train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

print("✅ 데이터셋 준비 및 토크나이징 완료")

config.json:   0%|          | 0.00/514 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/288 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Map:   0%|          | 0/15499 [00:00<?, ? examples/s]

Map:   0%|          | 0/3875 [00:00<?, ? examples/s]

✅ 데이터셋 준비 및 토크나이징 완료


In [5]:
# 정확도 계산 함수
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, predictions)}

# 모델 로드
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=4)

# 훈련 인자 설정
training_args = TrainingArguments(
    output_dir="./tamagotchi_checkpoints",
    report_to="mlflow",           # MLflow에 기록
    run_name="KcELECTRA_v2_Full_Retrain",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    load_best_model_at_end=True,
    fp16=True,                    # 코랩 T4 GPU 속도 향상
)

# 트레이너 정의
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

ElectraForSequenceClassification LOAD REPORT from: beomi/KcELECTRA-base
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
electra.embeddings.position_ids                   | UNEXPECTED | 
classifier.out_proj.bias                          | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 
classifier.dense.weight                           | MISSING    | 
classifier.dense.bias                             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoin

In [6]:
print("🚀 훈련을 시작합니다...")
trainer.train()

# 최종 모델 및 토크나이저 저장
trainer.save_model("./my_tamagotchi_model")
tokenizer.save_pretrained("./my_tamagotchi_model")
print("✅ 모델 저장 완료: ./my_tamagotchi_model")

🚀 훈련을 시작합니다...


Epoch,Training Loss,Validation Loss,Accuracy
1,0.275893,0.289854,0.916129
2,0.208835,0.258714,0.926194
3,0.128174,0.275607,0.934452


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['electra.embeddings.LayerNorm.weight', 'electra.embeddings.LayerNorm.bias', 'electra.encoder.layer.0.attention.output.LayerNorm.weight', 'electra.encoder.layer.0.attention.output.LayerNorm.bias', 'electra.encoder.layer.0.output.LayerNorm.weight', 'electra.encoder.layer.0.output.LayerNorm.bias', 'electra.encoder.layer.1.attention.output.LayerNorm.weight', 'electra.encoder.layer.1.attention.output.LayerNorm.bias', 'electra.encoder.layer.1.output.LayerNorm.weight', 'electra.encoder.layer.1.output.LayerNorm.bias', 'electra.encoder.layer.2.attention.output.LayerNorm.weight', 'electra.encoder.layer.2.attention.output.LayerNorm.bias', 'electra.encoder.layer.2.output.LayerNorm.weight', 'electra.encoder.layer.2.output.LayerNorm.bias', 'electra.encoder.layer.3.attention.output.LayerNorm.weight', 'electra.encoder.layer.3.attention.output.LayerNorm.bias', 'electra.encoder.layer.3.output.LayerNorm.weight', 'electra.encoder.layer.3.output.Laye

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ 모델 저장 완료: ./my_tamagotchi_model


In [8]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# 1. 저장된 모델과 토크나이저 불러오기
model_path = "./my_tamagotchi_model"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

# GPU 사용 설정 (코랩)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# 2. 감정 레이블 매핑 (훈련 때 설정한 순서와 동일해야 합니다)
# 기쁨:0, 슬픔:1, 화남:2, 평온:3
id2emotion = {0: "기쁨 😊", 1: "슬픔 😭", 2: "화남 💢", 3: "평온 😐"}

print("✨ 다마고치 감정 분석기가 준비되었습니다! (종료하려면 'exit' 입력)")

while True:
    user_input = input("나: ")
    if user_input.lower() == 'exit':
        break

    # 3. 입력 문장 전처리 및 추론
    inputs = tokenizer(user_input, return_tensors="pt", truncation=True, max_length=64).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        # 가장 높은 확률을 가진 인덱스(0,1,2,3) 찾기
        prediction = torch.argmax(logits, dim=-1).item()

    # 4. 결과 출력
    emotion = id2emotion.get(prediction, "알 수 없음")
    print(f"🐱 다마고치의 감정: {emotion}")
    print("-" * 30)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✨ 다마고치 감정 분석기가 준비되었습니다! (종료하려면 'exit' 입력)
나: 사랑해
🐱 다마고치의 감정: 화남 💢
------------------------------


KeyboardInterrupt: Interrupted by user

In [9]:
# 1. 모든 실험 결과 다시 불러오기
df_runs = mlflow.search_runs(experiment_names=["Tamagotchi_Emotion_Project"])

# 2. 실제로 어떤 컬럼들이 있는지 확인 (에러 방지용)
cols = df_runs.columns
target_cols = []

# run_name이 없으면 다른 이름으로 시도
for c in ['run_name', 'tags.mlflow.runName', 'metrics.eval_accuracy', 'metrics.eval_loss', 'start_time']:
    if c in cols:
        target_cols.append(c)

# 3. 데이터가 있을 때만 출력
if target_cols:
    display(df_runs[target_cols].sort_values(by='start_time', ascending=False))
else:
    print("기록된 데이터가 없습니다. 훈련이 정상적으로 끝났는지 확인해주세요.")

,tags.mlflow.runName,metrics.eval_accuracy,metrics.eval_loss,start_time
0,KcELECTRA_v2_Full_Retrain,0.934452,0.275607,2026-02-24 08:19:46.292000+00:00
